In [3]:
import pandas as pd
import re
import random
import os
from PIL import Image, ImageDraw, ImageFont
import cv2
import sys, os
import json
sys.path.append('../utils')
import text_utils
import image_utils
import numpy as np

salish_words, english_words = text_utils.load_wordlist("../sources/roots.csv", "../sources/english.txt")
for w in salish_words[:10]:
    print(repr(w))
def make_text_sample(salish_words, english_words):
    
    salish_sample = " ".join(random.sample(salish_words, k=random.randint(3,5)))
    english_sample = " ".join(random.sample(english_words, k=random.randint(3,5)))
    layouts = [
        salish_sample + random.choice(["."]),
        english_sample + random.choice(["."]),
        salish_sample + random.choice([".", "?"])  + "\n" + english_sample + random.choice([".", "?", "!", "…"])
    ]
    return random.choice(layouts)
os.makedirs("../new_synthetic/images", exist_ok=True)
font_cfg = json.load(open("../sources/fonts_config.json"))
all_fonts = font_cfg["Charis"] +font_cfg["Doulos"] +  font_cfg["NotoSans"]
def random_font():
    
    fpath = random.choice(all_fonts)
    size = random.randint(24, 64)
    return ImageFont.truetype(fpath, size=size)

def random_layout(img_w, img_h):
    """return margin and line spacing pattern"""
    margin = random.randint(20, 100)
    spacing = random.randint(10, 40)
    return margin, spacing

def render_text_block(text, font, img_w, img_h):
    img = Image.new("L", (img_w, img_h), color=255)  # grayscale
    draw = ImageDraw.Draw(img)
    margin, spacing = random_layout(img_w, img_h)
    y = margin
    for line in text.split("\n"):
        draw.text((margin, y), line, font=font, fill=0)  # black text
        bbox = draw.textbbox((0, 0), line, font=font)
        line_height = bbox[3] - bbox[1]
        y += line_height + spacing
    return img

# later:


entries = []
for i in range(5000):
    text = make_text_sample(salish_words, english_words)
    font = random_font()
    img = render_text_block(text, font, img_w=1400, img_h=400)
    fpath = f"../new_synthetic/images/sample_{i}.png"
    img.save(fpath)
    entries.append({'image': Image.open(fpath).convert("L"), "text": text})
df = pd.DataFrame(entries)
df.head()

'a'
'a'
'a·'
'aye'
'buc'
'ecbúcbucšn'
'sbúcbucšn'
'sbúcbucšnmš'
'bu·lí'
'bam'


,image,text
0,<PIL.Image.Image image mode=L size=1400x400 at...,ʕʷéleʔel snukʷmararímem qel'tmxʷɫwárč?\nstarts...
1,<PIL.Image.Image image mode=L size=1400x400 at...,wlumsmú sharíʔqsmš t'ixʷ as‑qʷáʔqʷeʔelstmi‑s h...
2,<PIL.Image.Image image mode=L size=1400x400 at...,tft directors operational enhancements quizzes.
3,<PIL.Image.Image image mode=L size=1400x400 at...,enxʷúlul'mxʷ ttm'i·xʷíl'š putéʔm česwíhm‑lš sc...
4,<PIL.Image.Image image mode=L size=1400x400 at...,monroe arab supplemental.


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2)
# we reset the indices to start from zero
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [5]:
ipa_extras = "äáä́éɛɛ́íιóúəɔụʕʔx̣šǰčɬ∤ɫʀᴇc̕l̕m̕n̕p̕q̕r̕ṛʀ̕t̕w̕y̕wertyuiopkjhgfdsazxcvbnmʷ"

extra_tokens = list(ipa_extras)

In [6]:
from transformers import AutoTokenizer, VisionEncoderDecoderModel, TrOCRProcessor

tokenizer = AutoTokenizer.from_pretrained("microsoft/trocr-base-printed")
tokenizer.add_tokens(extra_tokens)

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")
model.decoder.resize_token_embeddings(len(tokenizer))
text = "čəxʷən ɬə šə́yəxʷ"
encoding = processor.tokenizer(text, return_tensors="pt")
decoded = processor.tokenizer.decode(encoding.input_ids[0], skip_special_tokens=True)
print("Original:", text)
print("Decoded:", decoded)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Original: čəxʷən ɬə šə́yəxʷ
Decoded: čəxʷən ɬə šə́yəxʷ


In [7]:
# set special tokens used for creating the decoder_input_ids from the labels
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size

# set beam search parameters
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

In [8]:
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import os
from PIL import Image
from datasets import Dataset
dataset = Dataset.from_list(entries)

In [9]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

def preprocess(example):
    pixel_values = processor.feature_extractor(example["image"], return_tensors="pt").pixel_values[0]
    labels = tokenizer(example["text"], truncation=True, padding="max_length", max_length=128).input_ids
    example["pixel_values"] = pixel_values
    example["labels"] = labels
    return example

dataset = dataset.map(preprocess)

train_test = dataset.train_test_split(test_size=0.2)
train_dataset = train_test["train"]
val_dataset = train_test["test"]


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]/home/exouser/colrc-ocr-model/.venv/lib/python3.12/site-packages/transformers/models/trocr/processing_trocr.py:139: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(
Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


ValueError: Unsupported number of image dimensions: 2

In [ ]:

import evaluate
cer_metric = evaluate.load("cer")
def compute_metrics(pred):
    preds = pred.predictions
    labels = pred.label_ids

    # Replace -100 (ignored tokens) with padding token id
    labels[labels == -100] = processor.tokenizer.pad_token_id

    # Decode
    pred_str = processor.batch_decode(preds, skip_special_tokens=True)
    label_str = processor.batch_decode(labels, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


In [ ]:
from transformers import Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="steps",
    num_train_epochs=5,
    save_steps=500,
    eval_steps=500,
    logging_steps=500,
    output_dir="./trocr-salish-ocr",
    save_total_limit=2,
    remove_unused_columns=False,
)


In [ ]:

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)
 
trainer.train()


NameError: name 'Seq2SeqTrainer' is not defined

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)

# Plot training loss
plt.figure(figsize=(10,4))
plt.plot(logs["step"], logs["loss"], label="Training Loss", color='blue')
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

# Plot CER
if "eval_cer" in logs.columns:
    plt.figure(figsize=(10,4))
    plt.plot(logs["step"], logs["eval_cer"], label="Eval CER", color='red')
    plt.xlabel("Step")
    plt.ylabel("CER")
    plt.title("Evaluation Character Error Rate")
    plt.legend()
    plt.show()
else:
    print("No eval_cer found — check your eval_steps or compute_metrics setup.")


In [ ]:

# Paths
model_dir = "./trocr-salish/checkpoint-300"
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model = VisionEncoderDecoderModel.from_pretrained(model_dir)
img = Image.open("../test/workbook.png").convert("RGB")

pixel_values = processor(images=img, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("Predicted:", pred_text)


Predicted: 'iii iwii'ikkw  e ekhw'kw'hw 'enkkh'''w'' ''''?h   i  r  a  o  n  
